# Week 3 — Day 4: Compare correct vs. incorrect, log to MLflow

Tasks:
- Compare explanations across correct vs. incorrect predictions -- identify patterns
- Log XAI visualizations as MLflow artifacts attached to the corresponding model run

Deliverable: MLflow run updated with attached XAI artifact images; a comparison table of correct
vs. incorrect cases with explanation summaries.

### 1. Build the comparison table

In [1]:
import os
import pandas as pd

xai_dir = '../data/xai_examples'
curated_df = pd.read_csv(os.path.join(xai_dir, 'curated_examples.csv'))

# Attention-pattern notes from visual inspection of Day 2's Grad-CAM grid
# (data/xai_examples/gradcam/all_examples_gradcam.png) and Day 3's SHAP/LIME case studies.
ATTENTION_NOTES = {
    '1526181215_c1a94325ae.jpg': ('concentrated on subject', 'n/a -- correct'),
    '2541104331_a2d65cfa54.jpg': ('concentrated on subject', 'n/a -- correct'),
    '2741051940_89fb6b2cee.jpg': ('concentrated on subject', 'n/a -- correct (fixed by fine-tuning, was "pig" zero-shot)'),
    '3154813159_58a195236d.jpg': ('concentrated on subject', 'n/a -- correct'),
    '2708686056_1b8f356264.jpg': ('moderately concentrated', 'n/a -- correct'),
    '2326730558_75c20e5033.jpg': ('concentrated on subject', 'n/a -- correct'),
    '3091912922_0d6ebc8f6a.jpg': ('concentrated on subject', 'n/a -- correct'),
    '2332986053_864db84971.jpg': ('stuck on background texture, ignores face', 'decoding degeneration (repetition loop)'),
    '3561543598_3c1b572f9b.jpg': ('diffuse across whole scene', 'correct localization, imprecise description (vague "group of people")'),
    '3280672302_2967177653.jpg': ('concentrated on the two people', 'correct localization, wrong action ("posing" vs carrying)'),
    '3674168459_6245f4f658.jpg': ('concentrated on the two people', 'correct localization, close but generic description'),
    '3069786374_804e1123ac.jpg': ('concentrated on figure + basketball hoop in background', 'plausible misidentification -- real visual (hoop) + language ("playing ___") cues both point to "basketball"'),
    '1148238960_f8cacec2fc.jpg': ('scattered across multiple people/bikes', 'correct localization, wrong specific activity'),
    '3228069008_edb2961fc4.jpg': ('concentrated on the dogs', 'correct localization, wrong specific action'),
}

curated_df['attention_pattern'] = curated_df['image'].map(lambda x: ATTENTION_NOTES.get(x, ('-', '-'))[0])
curated_df['failure_type'] = curated_df['image'].map(lambda x: ATTENTION_NOTES.get(x, ('-', '-'))[1])

comparison_table = curated_df[['image', 'bucket', 'rougeL', 'predicted_caption', 'best_ground_truth', 'attention_pattern', 'failure_type']]
comparison_table.to_csv(os.path.join(xai_dir, 'correct_vs_incorrect_comparison.csv'), index=False)
comparison_table

,image,bucket,rougeL,predicted_caption,best_ground_truth,attention_pattern,failure_type
0,1526181215_c1a94325ae.jpg,good,0.923077,a brown dog jumping into the water.,a dog jumps into the water .,concentrated on subject,n/a -- correct
1,2541104331_a2d65cfa54.jpg,good,0.923077,a brown dog splashes through the water.,A brown dog splashing through water .,concentrated on subject,n/a -- correct
2,2741051940_89fb6b2cee.jpg,good,0.900000,a black and white dog is playing with a plasti...,A white dog is playing with a plastic bag .,concentrated on subject,"n/a -- correct (fixed by fine-tuning, was ""pig..."
3,3154813159_58a195236d.jpg,good,0.869565,a brown and white dog is standing on its hind ...,The brown and white dog is standing up on its ...,concentrated on subject,n/a -- correct
4,2708686056_1b8f356264.jpg,good,0.857143,a girl in a blue swimsuit is running into the ...,A girl in a blue swimsuit walks into the ocean .,moderately concentrated,n/a -- correct
5,2326730558_75c20e5033.jpg,good,0.857143,two brown dogs are playing in the snow.,Two dogs play in the snow .,concentrated on subject,n/a -- correct
6,3091912922_0d6ebc8f6a.jpg,good,0.818182,a brown dog is running with a yellow ball in i...,A dog runs with a yellow toy in its mouth .,concentrated on subject,n/a -- correct
7,2332986053_864db84971.jpg,poor,0.055556,a bearded bearded bearded bearded bearded bear...,A man wearing a helmet and sunglasses smiles .,"stuck on background texture, ignores face",decoding degeneration (repetition loop)
8,3561543598_3c1b572f9b.jpg,poor,0.222222,a man in a white shirt and a man in a black an...,A group of men wearing uniforms with hats gath...,diffuse across whole scene,"correct localization, imprecise description (v..."
9,3280672302_2967177653.jpg,poor,0.222222,a man and a woman are posing for the camera.,a boy carries a teenager on his back .,concentrated on the two people,"correct localization, wrong action (""posing"" v..."


### 2. Patterns identified

In [2]:
good_count = (curated_df['bucket'] == 'good').sum()
poor_count = (curated_df['bucket'] == 'poor').sum()
localization_correct_but_wrong_desc = curated_df[
    (curated_df['bucket'] == 'poor') & (curated_df['failure_type'].str.contains('correct localization', na=False))
]
print(f"Good: {good_count}, Poor: {poor_count}")
print(f"Of {poor_count} poor examples, {len(localization_correct_but_wrong_desc)} have CORRECT visual localization")
print(f"but wrong description -- i.e. a language/fine-grained-recognition failure, not a vision failure.")
print()
print("Only 1 example (the repetition-loop case) shows the vision encoder itself disengaging")
print("from the image -- and that coincides exactly with the decoding-degeneration failure mode.")

Good: 7, Poor: 7
Of 7 poor examples, 5 have CORRECT visual localization
but wrong description -- i.e. a language/fine-grained-recognition failure, not a vision failure.

Only 1 example (the repetition-loop case) shows the vision encoder itself disengaging
from the image -- and that coincides exactly with the decoding-degeneration failure mode.


### 3. Attach XAI artifacts to the winning model's MLflow run (config C)

In [3]:
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("flickr8k-image-captioning")

winning_run = [
    r for r in client.search_runs([experiment.experiment_id])
    if r.data.tags.get("mlflow.runName") == "C_lr5e-5_beam"
][0]
print(f"Attaching XAI artifacts to run: {winning_run.info.run_id} (C_lr5e-5_beam)")

artifact_paths = [
    '../data/processed/week3_curated_examples_grid.png',
    '../data/xai_examples/gradcam/all_examples_gradcam.png',
    '../data/xai_examples/gradcam/fixed_dog_case_per_word.png',
    '../data/xai_examples/gradcam/repetition_loop_case_per_word.png',
    '../data/xai_examples/shap_lime/lime_examples.png',
    '../data/xai_examples/correct_vs_incorrect_comparison.csv',
]

with mlflow.start_run(run_id=winning_run.info.run_id):
    for path in artifact_paths:
        if os.path.exists(path):
            mlflow.log_artifact(path, artifact_path="xai")
        else:
            print(f"  (skipped, not found: {path})")

print("Done. XAI artifacts now attached under the 'xai/' folder of this run.")

Attaching XAI artifacts to run: 6c47aaea395b444a8a2ec767a0efabef (C_lr5e-5_beam)


Done. XAI artifacts now attached under the 'xai/' folder of this run.
